# 01 — Pathology features (GPU)

Datasets: BRACS, SICAPv2, MHIST (from your Drive), CRC-VAL-HE-7K, NCT-CRC-HE-100K (20k subsample) and PCam test (20k subsample), downloaded from Zenodo. Encoders: the four v2 encoders plus eight public pathology foundation models (six are gated — see notebook 00). NCT-CRC-HE-100K is an 11.7 GB download; drop it from `ONLY`/`DATASETS` if you are short of time.

**Rough runtime (estimate, not measured):** first BRACS decode ~20 min (once, then cached on Drive); then roughly 4–6 GPU-hours on an L4 for all 12 encoders × 6 datasets, dominated by the four ~1B-parameter models. Spread over several sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
assert os.path.exists(os.path.join(CODE, "config.py")), "run 00_setup_and_prereg.ipynb first"
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import importlib, config as CFG
importlib.reload(CFG)
print("Drive:", CFG.DRIVE, "| project:", CFG.PROJECT)

## Refresh extraction code on Drive
Writes the latest extraction modules. These are not part of the registered analysis, so `prereg.json` stays valid.

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/imgcache.py
"""Dataset manifests and decoded 224x224 uint8 image caches.

Why decode once: v2 spent ~19 min per encoder just decoding BRACS RoIs
(139-173 MP each) over Drive's FUSE mount. Decoding each image once to a
224x224 uint8 array and letting every encoder read that array makes the
per-encoder cost GPU-bound. Preprocessing is identical to v2's CTransPath
extraction (Resize(224) shorter side, CenterCrop(224)); each encoder then
applies only its own mean/std normalisation.

Row order is defined ONCE, by the manifest, and every encoder's features are
written in that order. The manifest stores a hash of (uid, label) so the
analysis can assert alignment instead of assuming it.
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import subprocess  # noqa: F401  (kept for callers)
import time

import numpy as np

import config as CFG

IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Datasets fetched from the internet. Their decoded 224px caches are saved to
# Drive (CRC-VAL 1.1 GB, NCT 20k 3.0 GB, PCam 20k 3.0 GB) so a Colab reset never
# forces a re-download again. Set False if Drive space is tight.
DOWNLOADED = {"crc_val_7k", "nct_crc_100k", "pcam"}
PERSIST_DOWNLOADED = True


# ---------------------------------------------------------------------------
# small utilities
# ---------------------------------------------------------------------------

def _mk(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)


def manifest_hash(uids, labels) -> str:
    h = hashlib.sha256()
    for u, l in zip(uids, labels):
        h.update(f"{u}\t{int(l)}\n".encode())
    return h.hexdigest()


def stratified_cap(labels, cap, seed=CFG.SEED, groups=None):
    """Indices (sorted) of a stratified subsample of size `cap`.
    If groups are given, whole groups are kept or dropped together so that the
    grouped folds downstream stay meaningful."""
    labels = np.asarray(labels)
    n = len(labels)
    if cap is None or n <= cap:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    if groups is not None:
        groups = np.asarray(groups)
        ug = np.unique(groups)
        rng.shuffle(ug)
        keep, total = [], 0
        for g in ug:
            idx = np.where(groups == g)[0]
            if total + len(idx) > cap and total > 0:
                continue
            keep.append(idx)
            total += len(idx)
            if total >= cap * 0.98:
                break
        return np.sort(np.concatenate(keep))
    from sklearn.model_selection import train_test_split
    idx, _ = train_test_split(np.arange(n), train_size=cap, stratify=labels,
                              random_state=seed)
    return np.sort(idx)


def download(url, dst, retries=8, chunk=1 << 22):
    """Resumable download that reports progress IN THE NOTEBOOK.

    (The previous version used curl in a subprocess; Colab does not show a
    subprocess's progress bar, so an 11.7 GB download looked frozen.)
    A partial file is kept as <dst>.part and resumed with an HTTP Range request,
    so interrupting the cell loses nothing."""
    import requests
    if os.path.exists(dst) and os.path.getsize(dst) > 0 and not os.path.exists(dst + ".part"):
        return dst
    _mk(os.path.dirname(dst))
    part = dst + ".part"
    for attempt in range(retries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers,
                              allow_redirects=True) as r:
                if r.status_code == 416:                  # already complete
                    os.replace(part, dst)
                    return dst
                r.raise_for_status()
                if have and r.status_code != 206:          # server ignored Range: restart
                    have = 0
                clen = int(r.headers.get("Content-Length", 0) or 0)
                total = clen + have if clen else 0
                mode = "ab" if have else "wb"
                t0, last, got = time.time(), 0.0, have
                name = os.path.basename(dst)
                print(f"  downloading {name}: "
                      + (f"{total / 1e9:.2f} GB" if total > have else "size not reported")
                      + (f" (resuming at {have / 1e9:.2f} GB)" if have else ""), flush=True)
                with open(part, mode) as fh:
                    for blk in r.iter_content(chunk_size=chunk):
                        fh.write(blk)
                        got += len(blk)
                        now = time.time()
                        if now - last > 30:
                            rate = (got - have) / max(now - t0, 1e-9)
                            eta = (total - got) / max(rate, 1) / 60 if total else 0.0
                            print(f"    {got / 1e9:6.2f} / "
                                  + (f"{total / 1e9:.2f} GB  " if total > have else "? GB  ")
                                  + f"{rate / 1e6:6.1f} MB/s"
                                  + (f"  ETA {eta:5.1f} min" if total > have else ""), flush=True)
                            last = now
            if total and os.path.getsize(part) < total:
                raise IOError(f"incomplete: {os.path.getsize(part)} of {total} bytes")
            os.replace(part, dst)
            print(f"  downloaded {name} in {(time.time() - t0) / 60:.1f} min", flush=True)
            return dst
        except Exception as e:
            print(f"  download interrupted ({type(e).__name__}: {str(e)[:120]}); "
                  f"retry {attempt + 1}/{retries} in 15 s, resuming", flush=True)
            time.sleep(15)
    raise RuntimeError(f"download failed: {url}")


def _zip_tree(zp, inner):
    """Class-folder listing straight from the zip index (no extraction).
    Reproduces _folder_tree's ordering exactly: classes sorted, files sorted
    by name within a class. The manifest hash check guards this equivalence."""
    import zipfile
    by_cls = {}
    with zipfile.ZipFile(zp) as z:
        for n in z.namelist():
            parts = n.split("/")
            if len(parts) != 3 or parts[0] != inner or not parts[2]:
                continue
            if os.path.splitext(parts[2])[1].lower() not in IMG_EXT:
                continue
            by_cls.setdefault(parts[1], []).append(parts[2])
    classes = sorted(by_cls)
    root = os.path.join(CFG.LOCAL_DL, inner)
    files, labels = [], []
    for ci, c in enumerate(classes):
        for f in sorted(by_cls[c]):
            files.append(os.path.join(root, c, f))
            labels.append(ci)
    return classes, files, labels, root


def extract_needed(zp, files):
    """Extract only the members the manifest uses (20k of 100k for NCT)."""
    import zipfile
    missing = [f for f in files if not os.path.exists(f)]
    if not missing:
        return
    base = CFG.LOCAL_DL
    t0 = time.time()
    print(f"  extracting {len(missing)} of the zip's images that the manifest uses ...", flush=True)
    with zipfile.ZipFile(zp) as z:
        for i, f in enumerate(missing):
            z.extract(os.path.relpath(f, base).replace(os.sep, "/"), base)
            if (i + 1) % 5000 == 0:
                print(f"    {i + 1}/{len(missing)}", flush=True)
    print(f"  extracted in {(time.time() - t0) / 60:.1f} min", flush=True)


# ---------------------------------------------------------------------------
# manifests: each builder returns dict(uids, labels, groups|None, classes, reader)
#   reader = ("files", [paths]) | ("torchvision", spec) | ("h5", path, row_index)
# ---------------------------------------------------------------------------

def _bracs():
    root = CFG.BRACS_DIR
    classes = sorted(d for d in os.listdir(os.path.join(root, "train"))
                     if os.path.isdir(os.path.join(root, "train", d)))
    files, labels = [], []
    for split in ("train", "val"):                       # v2 order: train + val
        for ci, c in enumerate(classes):
            d = os.path.join(root, split, c)
            fs = sorted(f for f in os.listdir(d) if os.path.splitext(f)[1].lower() in IMG_EXT)
            files += [os.path.join(d, f) for f in fs]
            labels += [ci] * len(fs)
    pat = re.compile(r"^BRACS_(\d+)_")
    g = [pat.match(os.path.basename(f)) for f in files]
    if sum(m is None for m in g) > 0.01 * len(files):
        print("  [bracs] WARNING: WSI id not parseable from filenames -> ungrouped folds")
        groups = None
    else:
        groups = [("BRACS_" + m.group(1)) if m else os.path.basename(f) for m, f in zip(g, files)]
    uids = [os.path.relpath(f, root) for f in files]
    return dict(uids=uids, labels=labels, groups=groups, classes=classes, reader=("files", files))


def _sicapv2():
    import pandas as pd
    part = os.path.join(CFG.SICAP_DIR, "partition", "Test")
    cols = ["NC", "G3", "G4", "G5"]
    df = pd.concat([pd.read_excel(os.path.join(part, "Train.xlsx")),
                    pd.read_excel(os.path.join(part, "Test.xlsx"))], ignore_index=True)
    names = list(df["image_name"])
    labels = df[cols].values.argmax(1).tolist()
    groups = [n.split("_Block")[0] for n in names]
    files = [os.path.join(CFG.SICAP_DIR, "images", n) for n in names]
    return dict(uids=names, labels=labels, groups=groups, classes=cols, reader=("files", files))


def _mhist():
    import pandas as pd
    csvp = next((os.path.join(CFG.MHIST_DIR, c) for c in CFG.MHIST_CSV_CANDIDATES
                 if os.path.exists(os.path.join(CFG.MHIST_DIR, c))), None)
    if csvp is None:
        raise FileNotFoundError(f"no MHIST annotations csv in {CFG.MHIST_DIR}")
    df = pd.read_csv(csvp)
    classes = sorted(df["Majority Vote Label"].unique())          # ['HP', 'SSA']
    cmap = {c: i for i, c in enumerate(classes)}
    names = list(df["Image Name"])
    labels = df["Majority Vote Label"].map(cmap).tolist()
    files = [os.path.join(CFG.MHIST_DIR, "images", n) for n in names]
    return dict(uids=names, labels=labels, groups=None, classes=classes,
                reader=("files", files), extra={"partition": list(df["Partition"])})


def _folder_tree(root):
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    files, labels = [], []
    for ci, c in enumerate(classes):
        fs = sorted(f for f in os.listdir(os.path.join(root, c))
                    if os.path.splitext(f)[1].lower() in IMG_EXT)
        files += [os.path.join(root, c, f) for f in fs]
        labels += [ci] * len(fs)
    return classes, files, labels


def _zenodo_zip(fname, inner):
    zp = download(CFG.ZENODO_NCT.format(fname), os.path.join(CFG.LOCAL_DL, fname))
    classes, files, labels, root = _zip_tree(zp, inner)
    return dict(uids=[os.path.relpath(f, root) for f in files], labels=labels, groups=None,
                classes=classes, reader=("files", files), zip=zp)


def _crc_val_7k():
    return _zenodo_zip("CRC-VAL-HE-7K.zip", "CRC-VAL-HE-7K")


def _nct_crc_100k():
    return _zenodo_zip("NCT-CRC-HE-100K.zip", "NCT-CRC-HE-100K")


def _pcam():
    import gzip
    import pandas as pd
    out = {}
    for part in ("x", "y"):
        gz = download(CFG.ZENODO_PCAM.format(f"camelyonpatch_level_2_split_test_{part}.h5.gz"),
                      os.path.join(CFG.LOCAL_DL, f"pcam_test_{part}.h5.gz"))
        h5 = gz[:-3]
        if not os.path.exists(h5):
            print(f"  gunzip {os.path.basename(gz)} ...")
            with gzip.open(gz, "rb") as fi, open(h5 + ".tmp", "wb") as fo:
                shutil.copyfileobj(fi, fo, length=1 << 24)
            os.replace(h5 + ".tmp", h5)
        out[part] = h5
    meta = download(CFG.ZENODO_PCAM.format("camelyonpatch_level_2_split_test_meta.csv"),
                    os.path.join(CFG.LOCAL_DL, "pcam_test_meta.csv"))
    import h5py
    with h5py.File(out["y"], "r") as f:
        labels = np.asarray(f["y"]).reshape(-1).astype(int).tolist()
    wsi = list(pd.read_csv(meta)["wsi"])
    assert len(wsi) == len(labels), "PCam meta / label length mismatch"
    n = len(labels)
    return dict(uids=[f"pcam_test_{i}" for i in range(n)], labels=labels, groups=wsi,
                classes=["normal", "tumor"], reader=("h5", out["x"], list(range(n))))


class _HFImages:
    """A Hugging Face image dataset exposed like a torchvision dataset
    (__getitem__ -> (PIL image, label)), used as a fallback mirror."""

    def __init__(self, hf_id, split):
        from datasets import load_dataset
        self.ds = load_dataset(hf_id, split=split)
        self._labels = [int(v) for v in self.ds["label"]]
        self.classes = list(self.ds.features["label"].names)
        self.source = f"huggingface:{hf_id}:{split}"

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, i):
        r = self.ds[int(i)]
        return r["image"], int(r["label"])


# Mirrors used only when the official torchvision host fails. A manifest built
# from a mirror records it (reader name "<dataset>_hf"), so every later session
# decodes from the same source and row order.
HF_MIRRORS = {"gtsrb": ("tanganke/gtsrb", "train")}   # 26,640 images, 43 classes (= torchvision train)


def _tv(name):
    """torchvision datasets: returns (dataset_without_transform, labels)."""
    if name.endswith("_hf"):
        hf_id, split = HF_MIRRORS[name[:-3]]
        ds = _HFImages(hf_id, split)
        return ds, ds._labels
    from torchvision import datasets as D
    root = os.path.join(CFG.LOCAL_DL, "tv")
    _mk(root)
    if name == "dtd":
        ds = D.DTD(root, split="train", download=True)
    elif name == "aircraft":
        ds = D.FGVCAircraft(root, split="train", download=True)
    elif name == "eurosat":
        ds = D.EuroSAT(root, download=True)
    elif name == "pets":
        ds = D.OxfordIIITPet(root, split="trainval", download=True)
    elif name == "food101":
        ds = D.Food101(root, split="train", download=True)
    elif name == "cifar100":
        ds = D.CIFAR100(root, train=True, download=True)
    elif name == "gtsrb":
        ds = D.GTSRB(root, split="train", download=True)
    elif name == "country211":
        ds = D.Country211(root, split="train", download=True)
    else:
        raise KeyError(name)
    labels = tv_labels(ds)
    return ds, labels


def tv_labels(ds):
    for attr in ("targets", "_labels"):
        if hasattr(ds, attr):
            return [int(v) for v in getattr(ds, attr)]
    if hasattr(ds, "_samples"):
        return [int(s[1]) for s in ds._samples]
    print("  (labels not exposed; iterating once)")
    return [int(ds[i][1]) for i in range(len(ds))]


def _tv_manifest(name):
    src = name
    try:
        ds, labels = _tv(name)
    except Exception as e:
        if name not in HF_MIRRORS:
            raise
        print(f"  [{name}] official torchvision download failed ({type(e).__name__}: "
              f"{str(e)[:100]}); using mirror {HF_MIRRORS[name][0]}", flush=True)
        src = name + "_hf"
        ds, labels = _tv(src)
    classes = [str(c) for c in getattr(ds, "classes", sorted(set(labels)))]
    return dict(uids=[f"{src}_{i}" for i in range(len(labels))], labels=labels, groups=None,
                classes=classes, reader=("torchvision", src, list(range(len(labels)))),
                extra_source=getattr(ds, "source", f"torchvision:{name}"))


BUILDERS = {
    "bracs": _bracs, "sicapv2": _sicapv2, "mhist": _mhist,
    "crc_val_7k": _crc_val_7k, "nct_crc_100k": _nct_crc_100k, "pcam": _pcam,
    **{k: (lambda k=k: _tv_manifest(k)) for k in
       ("dtd", "aircraft", "eurosat", "pets", "food101", "cifar100", "gtsrb", "country211")},
}


def dataset_cfg(name):
    for dsets in (CFG.PATHOLOGY_DATASETS, CFG.VISION_DATASETS):
        if name in dsets:
            return dsets[name]
    raise KeyError(name)


def build_manifest(name, force=False):
    """Build (or load) the manifest; applies the stratified cap. Saved to Drive."""
    _mk(CFG.MANIFEST_DIR)
    jp = os.path.join(CFG.MANIFEST_DIR, f"{name}.json")
    if os.path.exists(jp) and not force:
        with open(jp) as fh:
            man = json.load(fh)
        return man
    t0 = time.time()
    m = BUILDERS[name]()
    labels = np.asarray(m["labels"], int)
    groups = None if m["groups"] is None else np.asarray(m["groups"])
    cap = dataset_cfg(name)["cap"]
    keep = stratified_cap(labels, cap, groups=groups)
    reader = list(m["reader"])
    # subset the reader's per-row source list (last element)
    reader[-1] = [reader[-1][i] for i in keep]
    if m.get("zip"):
        extract_needed(m["zip"], reader[-1])
    man = dict(
        name=name,
        n=int(len(keep)), n_before_cap=int(len(labels)), cap=cap, seed=CFG.SEED,
        classes=list(m["classes"]),
        uids=[m["uids"][i] for i in keep],
        labels=labels[keep].tolist(),
        groups=None if groups is None else groups[keep].tolist(),
        reader=reader,
        class_counts=np.bincount(labels[keep], minlength=len(m["classes"])).tolist(),
        n_groups=None if groups is None else int(len(np.unique(groups[keep]))),
        built=time.strftime("%Y-%m-%d %H:%M:%S"),
    )
    if "extra" in m:
        man["extra"] = {k: [v[i] for i in keep] for k, v in m["extra"].items()}
    if "extra_source" in m:
        man["source"] = m["extra_source"]
    man["hash"] = manifest_hash(man["uids"], man["labels"])
    with open(jp, "w") as fh:
        json.dump(man, fh)
    with open(os.path.join(CFG.MANIFEST_DIR, f"{name}.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["row", "uid", "label", "group"])
        for i, (u, l) in enumerate(zip(man["uids"], man["labels"])):
            w.writerow([i, u, l, "" if man["groups"] is None else man["groups"][i]])
    print(f"  [{name}] manifest n={man['n']} (of {man['n_before_cap']}) "
          f"classes={len(man['classes'])} groups={man['n_groups']} "
          f"({time.time() - t0:.0f}s)")
    return man


# ---------------------------------------------------------------------------
# decoding to the uint8 cache
# ---------------------------------------------------------------------------

class _Decoder:
    """torch Dataset yielding uint8 (224, 224, 3) arrays for a manifest."""

    def __init__(self, man, size=CFG.IMAGE_SIZE):
        self.reader = man["reader"]
        self.size = size
        self._h5 = None
        self._tv = None

    def __len__(self):
        return len(self.reader[-1])

    def _tf(self, img):
        from PIL import Image
        img = img.convert("RGB")
        w, h = img.size
        s = self.size / min(w, h)
        nw, nh = max(self.size, round(w * s)), max(self.size, round(h * s))
        img = img.resize((nw, nh), Image.BILINEAR, reducing_gap=3.0)
        l, t = (nw - self.size) // 2, (nh - self.size) // 2
        return np.asarray(img.crop((l, t, l + self.size, t + self.size)), dtype=np.uint8)

    def __getitem__(self, i):
        from PIL import Image
        Image.MAX_IMAGE_PIXELS = None
        kind = self.reader[0]
        if kind == "files":
            img = Image.open(self.reader[1][i])
            try:
                img.draft("RGB", (2 * self.size, 2 * self.size))   # JPEG only; no-op otherwise
            except Exception:
                pass
            return self._tf(img)
        if kind == "h5":
            import h5py
            if self._h5 is None:
                self._h5 = h5py.File(self.reader[1], "r")["x"]
            return self._tf(Image.fromarray(np.asarray(self._h5[self.reader[2][i]])))
        if kind == "torchvision":
            if self._tv is None:
                self._tv, _ = _tv(self.reader[1])
            img, _ = self._tv[self.reader[2][i]]
            return self._tf(img)
        raise ValueError(kind)


def _sources_present(man, n_check=25):
    kind = man["reader"][0]
    if kind == "files":
        paths = man["reader"][1]
        step = max(1, len(paths) // n_check)
        return all(os.path.exists(p) for p in paths[::step]) and os.path.exists(paths[-1])
    if kind == "h5":
        return os.path.exists(man["reader"][1])
    return True          # torchvision: checked by ensure_sources itself


def ensure_sources(name, man):
    """Make sure the raw images behind a saved manifest exist in THIS session.

    Manifests live on Drive and survive a Colab restart; downloaded images live
    in /content and do not. Without this, a resumed run tries to decode files
    that are gone (the CRC-VAL-HE-7K FileNotFoundError). Re-running the builder
    re-downloads the data; the rebuilt manifest must hash-match the saved one,
    so the row order cannot silently change between sessions."""
    kind = man["reader"][0]
    if kind == "torchvision":
        # download ONCE in the main process; otherwise every DataLoader worker
        # would start its own download of the same archive
        _tv(man["reader"][1])
        return
    if _sources_present(man):
        return
    if name in ("bracs", "sicapv2", "mhist"):
        raise FileNotFoundError(f"[{name}] source images missing on Drive: e.g. "
                                f"{man['reader'][1][0]} -- check the PATHS block in config.py")
    print(f"  [{name}] source images not in this session (runtime was reset); "
          f"downloading again ...", flush=True)
    m = BUILDERS[name]()
    labels = np.asarray(m["labels"], int)
    groups = None if m["groups"] is None else np.asarray(m["groups"])
    keep = stratified_cap(labels, dataset_cfg(name)["cap"], groups=groups)
    uids = [m["uids"][i] for i in keep]
    h = manifest_hash(uids, labels[keep].tolist())
    if h != man["hash"]:
        raise RuntimeError(f"[{name}] re-downloaded data does not reproduce the saved manifest "
                           f"(hash {h[:12]} vs {man['hash'][:12]}); refusing to mix row orders")
    src = list(m["reader"])
    src[-1] = [src[-1][i] for i in keep]
    if src[-1] != man["reader"][-1]:
        raise RuntimeError(f"[{name}] source paths changed after re-download")
    if m.get("zip"):
        extract_needed(m["zip"], src[-1])
    if not _sources_present(man):
        raise FileNotFoundError(f"[{name}] sources still missing after re-download")
    print(f"  [{name}] sources restored; manifest hash verified", flush=True)


def _cache_paths(name):
    return (os.path.join(CFG.LOCAL_IMG, f"{name}.npy"),
            os.path.join(CFG.LOCAL_IMG, f"{name}.progress.json"),
            os.path.join(CFG.IMG_DRIVE_DIR, f"{name}.npy"))


def _is_complete(npy, man):
    meta = npy + ".meta.json"
    if not (os.path.exists(npy) and os.path.exists(meta)):
        return False
    with open(meta) as fh:
        m = json.load(fh)
    return m.get("hash") == man["hash"] and m.get("complete") is True


def ensure_images(name, workers=8, chunk=1024):
    """Return a read-only memmap (n, 224, 224, 3) uint8 for dataset `name`.

    Order of preference: complete local cache -> complete Drive cache (copied
    to local) -> decode now (resumable chunk by chunk)."""
    import torch
    from torch.utils.data import DataLoader
    _mk(CFG.LOCAL_IMG, CFG.IMG_DRIVE_DIR)
    man = build_manifest(name)
    local, prog, drive = _cache_paths(name)
    if _is_complete(local, man):
        return np.load(local, mmap_mode="r"), man
    if _is_complete(drive, man):
        print(f"  [{name}] copying image cache from Drive ...")
        shutil.copy(drive, local)
        shutil.copy(drive + ".meta.json", local + ".meta.json")
        return np.load(local, mmap_mode="r"), man

    ensure_sources(name, man)
    n, S = man["n"], CFG.IMAGE_SIZE
    if os.path.exists(local) and os.path.exists(prog):
        arr = np.load(local, mmap_mode="r+")
        with open(prog) as fh:
            done = set(json.load(fh)["done"])
    else:
        arr = np.lib.format.open_memmap(local, mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
        done = set()
    chunks = [(s, min(s + chunk, n)) for s in range(0, n, chunk)]
    todo = [i for i in range(len(chunks)) if i not in done]
    print(f"  [{name}] decoding {n} images: {len(todo)}/{len(chunks)} chunks to go")
    dec = _Decoder(man)
    t0 = time.time()
    for k, ci in enumerate(todo):
        a, b = chunks[ci]
        sub = torch.utils.data.Subset(dec, list(range(a, b)))
        dl = DataLoader(sub, batch_size=64, num_workers=workers, shuffle=False,
                        collate_fn=lambda xs: np.stack(xs))
        pos = a
        for batch in dl:
            arr[pos:pos + len(batch)] = batch
            pos += len(batch)
        assert pos == b
        arr.flush()
        done.add(ci)
        with open(prog, "w") as fh:
            json.dump({"done": sorted(done)}, fh)
        el = time.time() - t0
        eta = el / (k + 1) * (len(todo) - k - 1) / 60
        print(f"    chunk {ci + 1}/{len(chunks)}  {b}/{n}  ETA {eta:5.1f} min", flush=True)
    del arr
    # sanity: no all-zero images (an undecoded row would be all zeros)
    arr = np.load(local, mmap_mode="r")
    zero = [i for i in range(0, n, max(1, n // 500)) if not arr[i].any()]
    if zero:
        raise RuntimeError(f"{len(zero)} sampled rows are all-zero, e.g. {zero[:5]}")
    with open(local + ".meta.json", "w") as fh:
        json.dump({"hash": man["hash"], "complete": True, "n": n}, fh)
    if os.path.exists(prog):
        os.remove(prog)
    if dataset_cfg(name).get("persist") or (PERSIST_DOWNLOADED and name in DOWNLOADED):
        print(f"  [{name}] persisting image cache to Drive ({arr.nbytes / 1e9:.2f} GB) ...")
        shutil.copy(local, drive + ".tmp")
        os.replace(drive + ".tmp", drive)
        shutil.copy(local + ".meta.json", drive + ".meta.json")
    return arr, man


def drop_local_images(name):
    local, prog, _ = _cache_paths(name)
    for p in (local, local + ".meta.json", prog):
        if os.path.exists(p):
            os.remove(p)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/encoders.py
"""Encoder construction and frozen-feature extraction.

Every encoder is wrapped so that it takes a uint8 batch (B, 224, 224, 3) and
returns float32 features (B, d). Each wrapper records what it actually did
(input size, mean/std, pooling, parameter count, weight checksum) so the paper
can state it and a reviewer can check it.

Load safety. The v1 CTransPath failure was a checkpoint silently loaded into the
wrong module. For hub models we rely on the library's strict load (timm and
transformers both raise on missing weights unless told otherwise) and we add a
second guard after extraction: feature statistics (effective rank, dead
dimensions, finiteness) and a quick probe accuracy per encoder are tabulated so
an encoder that "works" but is degenerate stands out. CTransPath, whose
checkpoint we load ourselves, runs in a separate process with the original
timm 0.5.4 and an explicit full-load assertion (ctranspath_worker.py).
"""

from __future__ import annotations

import hashlib
import os

import numpy as np

import config as CFG

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def hf_login():
    """Use HF_TOKEN from Colab secrets (key icon in the left bar) if present."""
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
        try:
            from huggingface_hub import login
            login(token=tok, add_to_git_credential=False)
        except Exception as e:
            print("  HF login failed:", e)
        return True
    print("  No HF_TOKEN found: gated encoders (UNI, UNI2-h, Virchow2, H-optimus-0, "
          "Prov-GigaPath, Hibou-B) will be skipped.")
    return False


def _timm_kwargs(spec):
    import torch
    kw = spec.get("kw") or {}
    if kw == "UNI2_H":
        import timm
        return dict(img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5,
                    embed_dim=1536, mlp_ratio=2.66667 * 2, num_classes=0, no_embed_class=True,
                    mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU,
                    reg_tokens=8, dynamic_img_size=True)
    if kw == "SWIGLU":
        import timm
        return dict(mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU)
    return dict(kw)


def _shim_transformers_onnx():
    """transformers >= 5 removed `transformers.onnx`. Hibou-B's remote
    configuration file still imports `OnnxConfig` from it, only to define an
    ONNX-export helper class that feature extraction never uses. Register a
    stand-in module so that import succeeds. Does nothing if the real module
    exists. This changes no weights and no forward computation."""
    import importlib
    import sys
    import types
    try:
        importlib.import_module("transformers.onnx")
        return
    except Exception:
        pass
    mod = types.ModuleType("transformers.onnx")

    class OnnxConfig:                                   # never instantiated for inference
        default_onnx_opset = 14

        def __init__(self, *a, **k):
            raise RuntimeError("ONNX export is not available (stand-in module)")

    mod.OnnxConfig = OnnxConfig
    mod.OnnxConfigWithPast = OnnxConfig
    mod.OnnxSeq2SeqConfigWithPast = OnnxConfig
    sys.modules["transformers.onnx"] = mod
    try:
        import transformers
        transformers.onnx = mod
    except Exception:
        pass


class Encoder:
    def __init__(self, name, spec, device):
        self.name, self.spec, self.device = name, spec, device
        self.kind = spec["kind"]
        self.info = {"name": name, "kind": self.kind, "id": spec.get("id")}
        if self.kind == "timm":
            self._build_timm()
        elif self.kind == "hf":
            self._build_hf()
        else:
            raise ValueError(f"{self.kind} is not built in-process")
        self.model.eval().to(device)
        p = sum(t.numel() for t in self.model.parameters())
        self.info["n_params"] = int(p)
        self.info["weight_checksum"] = self._checksum()

    # -------------------------------------------------------------- builders
    def _build_timm(self):
        import timm
        kw = _timm_kwargs(self.spec)
        kw.setdefault("num_classes", 0)
        m = timm.create_model(self.spec["id"], pretrained=True, **kw)
        cfg = timm.data.resolve_model_data_config(m)
        native = tuple(cfg.get("input_size", (3, 224, 224)))[-1]
        if native != CFG.IMAGE_SIZE and "dynamic_img_size" not in kw and "img_size" not in kw:
            # e.g. DINOv2 defaults to 518. Rebuild at 224; timm resamples the
            # position embedding from the checkpoint. Recorded in self.info.
            # CNNs (ResNet, ConvNeXt) take any input size and reject these ViT-only
            # arguments; they are simply run at 224 as built.
            try:
                m = timm.create_model(self.spec["id"], pretrained=True,
                                      img_size=CFG.IMAGE_SIZE, **kw)
                self.info["resized_from"] = native
            except TypeError:
                try:
                    m = timm.create_model(self.spec["id"], pretrained=True,
                                          dynamic_img_size=True, **kw)
                    self.info["dynamic_from"] = native
                except TypeError:
                    self.info["run_at_224_native"] = native
        self.model = m
        self.mean = tuple(self.spec.get("mean") or cfg.get("mean") or IMAGENET_MEAN)
        self.std = tuple(self.spec.get("std") or cfg.get("std") or IMAGENET_STD)
        self.pool = self.spec.get("pool", "default")
        self.info.update(mean=self.mean, std=self.std, pool=self.pool,
                         timm_version=timm.__version__,
                         num_prefix_tokens=int(getattr(m, "num_prefix_tokens", 0) or 0))

    def _build_hf(self):
        import transformers
        from transformers import AutoImageProcessor, AutoModel
        trc = bool(self.spec.get("trust_remote_code", False))
        if trc:
            _shim_transformers_onnx()
        self.model = AutoModel.from_pretrained(self.spec["id"], trust_remote_code=trc)
        try:
            proc = AutoImageProcessor.from_pretrained(self.spec["id"], trust_remote_code=trc)
            self.mean = tuple(proc.image_mean)
            self.std = tuple(proc.image_std)
        except Exception:
            self.mean, self.std = IMAGENET_MEAN, IMAGENET_STD
        self.pool = self.spec.get("pool", "cls")
        self.info.update(mean=self.mean, std=self.std, pool=self.pool,
                         transformers_version=transformers.__version__)

    def _checksum(self):
        h = hashlib.sha256()
        for i, t in enumerate(self.model.state_dict().values()):
            if i % 7 == 0 and t.numel():
                h.update(np.ascontiguousarray(t.detach().float().flatten()[:64].cpu().numpy()).tobytes())
        return h.hexdigest()[:16]

    # -------------------------------------------------------------- forward
    def __call__(self, x_uint8):
        """x_uint8: torch uint8 tensor (B, H, W, 3) on any device."""
        import torch
        x = x_uint8.to(self.device, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
        mean = torch.tensor(self.mean, device=self.device).view(1, 3, 1, 1)
        std = torch.tensor(self.std, device=self.device).view(1, 3, 1, 1)
        x = (x - mean) / std
        use_amp = self.device.startswith("cuda")
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            if self.kind == "timm":
                if self.pool == "cls+meanpatch":
                    tok = self.model.forward_features(x)
                    npt = self.info["num_prefix_tokens"] or 1
                    f = torch.cat([tok[:, 0], tok[:, npt:].mean(1)], dim=-1)
                else:
                    f = self.model(x)
                    if f.ndim == 3:
                        f = f.mean(1)
                    elif f.ndim == 4:
                        f = f.mean((2, 3))
            else:
                out = self.model(pixel_values=x)
                if self.pool == "pooler" and getattr(out, "pooler_output", None) is not None:
                    f = out.pooler_output
                else:
                    f = out.last_hidden_state[:, 0]
        return f.float().cpu().numpy()


def build(name, spec, device):
    return Encoder(name, spec, device)


# ---------------------------------------------------------------------------
# guards
# ---------------------------------------------------------------------------

def feature_stats(F):
    F = np.asarray(F, np.float64)
    Fc = F - F.mean(0, keepdims=True)
    s = np.linalg.svd(Fc[: min(len(Fc), 4000)], compute_uv=False)
    p = s ** 2 / max((s ** 2).sum(), 1e-12)
    eff_rank = float(np.exp(-(p * np.log(p + 1e-12)).sum()))
    return {"n": int(F.shape[0]), "dim": int(F.shape[1]),
            "finite": bool(np.isfinite(F).all()),
            "mean_norm": float(np.linalg.norm(F, axis=1).mean()),
            "eff_rank": eff_rank, "eff_rank_frac": eff_rank / F.shape[1],
            "dead_dims_frac": float((F.std(0) < 1e-8).mean())}

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/extract.py
"""Resumable feature extraction.

Layout on Drive:
    features/<dataset>/<encoder>.npy        (n, d) float32, rows in manifest order
    features/<dataset>/<encoder>.json       provenance: encoder info, manifest hash, stats
    features/<dataset>/_shards_<encoder>/   temporary; deleted after assembly

A Colab disconnect loses at most one shard (SHARD_ROWS rows) of one encoder.
Re-running the notebook cell resumes exactly where it stopped.
"""

from __future__ import annotations

import glob
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np

import config as CFG
import encoders as E

TIMM054_DIR = os.environ.get("V3_TIMM054", "/content/timm054")


def _paths(ds, enc):
    d = os.path.join(CFG.FEAT_DIR, ds)
    return d, os.path.join(d, f"{enc}.npy"), os.path.join(d, f"{enc}.json"), \
        os.path.join(d, f"_shards_{enc}")


def is_done(ds, enc, man):
    _, npy, meta, _ = _paths(ds, enc)
    if not (os.path.exists(npy) and os.path.exists(meta)):
        return False
    with open(meta) as fh:
        m = json.load(fh)
    return m.get("manifest_hash") == man["hash"] and m.get("complete") is True


def _shard_ranges(n):
    return [(s, min(s + CFG.SHARD_ROWS, n)) for s in range(0, n, CFG.SHARD_ROWS)]


def _assemble(ds, enc, man, info, t_start):
    d, npy, meta, sh = _paths(ds, enc)
    ranges = _shard_ranges(man["n"])
    parts = [np.load(os.path.join(sh, f"shard_{i:04d}.npy")) for i in range(len(ranges))]
    F = np.concatenate(parts).astype(CFG.FEATURE_DTYPE)
    assert F.shape[0] == man["n"], f"{F.shape[0]} rows != manifest {man['n']}"
    stats = E.feature_stats(F)
    if not stats["finite"]:
        raise RuntimeError(f"{ds}/{enc}: non-finite features; not saving")
    np.save(npy + ".tmp.npy", F)
    os.replace(npy + ".tmp.npy", npy)
    with open(meta, "w") as fh:
        json.dump({"dataset": ds, "encoder": enc, "manifest_hash": man["hash"],
                   "shape": list(F.shape), "dtype": CFG.FEATURE_DTYPE, "complete": True,
                   "encoder_info": info, "stats": stats,
                   "minutes": round((time.time() - t_start) / 60, 1),
                   "saved": time.strftime("%Y-%m-%d %H:%M:%S")}, fh, indent=1, default=str)
    shutil.rmtree(sh, ignore_errors=True)
    return stats


def extract_inprocess(ds, enc_name, spec, images, man, device, enc_obj=None):
    import torch
    d, npy, meta, sh = _paths(ds, enc_name)
    t0 = time.time()
    enc = enc_obj or E.build(enc_name, spec, device)   # build first: a failed load leaves no trace
    os.makedirs(sh, exist_ok=True)
    ranges = _shard_ranges(man["n"])
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(sh, f"shard_{i:04d}.npy"))]
    print(f"    {enc_name}: {enc.info.get('n_params', 0) / 1e6:.0f}M params, "
          f"{len(todo)}/{len(ranges)} shards to go", flush=True)
    for k, i in enumerate(todo):
        a, b = ranges[i]
        out = []
        for s in range(a, b, CFG.EXTRACT_BATCH):
            x = torch.from_numpy(np.ascontiguousarray(images[s:min(s + CFG.EXTRACT_BATCH, b)]))
            out.append(enc(x))
        np.save(os.path.join(sh, f"shard_{i:04d}.npy"), np.concatenate(out).astype(np.float32))
        el = time.time() - t0
        print(f"      shard {i + 1}/{len(ranges)}  {b}/{man['n']}  "
              f"{(b - a) * (k + 1) / max(el, 1e-9):6.0f} img/s  "
              f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)
    info = dict(enc.info)
    stats = _assemble(ds, enc_name, man, info, t0)
    return stats


# ---------------------------------------------------------------------------
# CTransPath: separate process with the original timm 0.5.4
# ---------------------------------------------------------------------------

def ensure_timm054():
    if os.path.exists(os.path.join(TIMM054_DIR, "timm", "__init__.py")):
        return
    print("  installing timm==0.5.4 into an isolated directory (for CTransPath only) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "--target", TIMM054_DIR, "timm==0.5.4"], check=True)


def _run_streaming(cmd, env):
    """Run a worker and echo its output line by line INTO THE NOTEBOOK.
    (Colab does not display a child process's own stdout/stderr.)"""
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        line = line.rstrip("\n")
        tail = (tail + [line])[-40:]
        if ("Warning" in line or "warn(" in line) and "Error" not in line:
            continue
        print(line, flush=True)
    p.wait()
    return p.returncode, "\n".join(tail)


# Hugging Face models whose remote code needs an older transformers than Colab's.
# They run in a separate process against an isolated install (like CTransPath).
ISOLATED_HF = {"hibou_b": "4.46.3"}
OLD_TF_DIR = os.environ.get("V3_OLD_TF", "/content/transformers_4_46")


def ensure_old_transformers(version="4.46.3"):
    marker = os.path.join(OLD_TF_DIR, f".installed_{version}")
    if os.path.exists(marker):
        return
    print(f"  installing transformers=={version} into an isolated directory "
          f"(for remote-code models only) ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "--target", OLD_TF_DIR, f"transformers=={version}",
                    "tokenizers==0.20.3", "huggingface_hub==0.26.5"], check=True)
    open(marker, "w").close()


def extract_hf_isolated(ds, enc_name, spec, images_path, man, device):
    d, npy, meta, sh = _paths(ds, enc_name)
    os.makedirs(sh, exist_ok=True)
    ensure_old_transformers(ISOLATED_HF[enc_name])
    t0 = time.time()
    env = dict(os.environ)
    env["PYTHONPATH"] = OLD_TF_DIR + os.pathsep + CFG.CODE_DIR
    env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
    cmd = [sys.executable, os.path.join(CFG.CODE_DIR, "hf_worker.py"), "--model", spec["id"],
           "--pool", spec.get("pool", "cls"), "--images", images_path, "--shards", sh,
           "--shard_rows", str(CFG.SHARD_ROWS), "--batch", str(CFG.EXTRACT_BATCH),
           "--device", device]
    if spec.get("trust_remote_code"):
        cmd.append("--trust_remote_code")
    rc, tail = _run_streaming(cmd, env)
    if rc != 0:
        if not any(f.startswith("shard_") for f in os.listdir(sh)):
            shutil.rmtree(sh, ignore_errors=True)        # nothing done: leave no "partial"
        raise RuntimeError(f"{enc_name} worker failed: {tail.splitlines()[-1] if tail else rc}")
    with open(os.path.join(sh, "load_report.json")) as fh:
        rep = json.load(fh)
    info = {"name": enc_name, "kind": "hf-isolated", "id": spec["id"],
            "mean": rep["mean"], "std": rep["std"], "pool": rep["pool"],
            "n_params": rep["n_params"], "load_report": rep}
    return _assemble(ds, enc_name, man, info, t0)


def extract_ctranspath(ds, images_path, man, device):
    d, npy, meta, sh = _paths(ds, "ctranspath")
    os.makedirs(sh, exist_ok=True)
    if not os.path.exists(CFG.CTRANSPATH_CKPT):
        raise FileNotFoundError(CFG.CTRANSPATH_CKPT)
    ensure_timm054()
    t0 = time.time()
    env = dict(os.environ)
    env["PYTHONPATH"] = TIMM054_DIR + os.pathsep + CFG.CODE_DIR
    worker = os.path.join(CFG.CODE_DIR, "ctranspath_worker.py")
    cmd = [sys.executable, worker, "--images", images_path, "--ckpt", CFG.CTRANSPATH_CKPT,
           "--shards", sh, "--shard_rows", str(CFG.SHARD_ROWS),
           "--batch", str(CFG.EXTRACT_BATCH), "--device", device]
    rc, _ = _run_streaming(cmd, env)
    if rc != 0:
        raise RuntimeError("CTransPath worker failed (see output above); nothing saved")
    with open(os.path.join(sh, "load_report.json")) as fh:
        rep = json.load(fh)
    info = {"name": "ctranspath", "kind": "ctranspath", "id": CFG.CTRANSPATH_CKPT,
            "mean": E.IMAGENET_MEAN, "std": E.IMAGENET_STD, "pool": "avg (timm 0.5.4 head)",
            "load_report": rep}
    return _assemble(ds, "ctranspath", man, info, t0)


# ---------------------------------------------------------------------------
# driver
# ---------------------------------------------------------------------------

def run_domain(datasets, encoders, device="cuda", only=None, skip_encoders=()):
    """Extract every (dataset, encoder) that is not already complete."""
    import imgcache as IC
    log = []
    E.hf_login()
    for ds in datasets:
        if only and ds not in only:
            continue
        pending = None
        print(f"\n=== {ds} ===", flush=True)
        try:
            man = IC.build_manifest(ds)
        except Exception as e:
            print(f"  !! manifest failed: {type(e).__name__}: {e}")
            log.append({"dataset": ds, "encoder": None, "status": f"manifest failed: {e}"})
            continue
        pending = [e for e in encoders if e not in skip_encoders and not is_done(ds, e, man)]
        if not pending:
            print("  all encoders complete")
            continue
        images, man = IC.ensure_images(ds)
        images_path = IC._cache_paths(ds)[0]
        for enc_name in pending:
            spec = encoders[enc_name]
            t = time.time()
            try:
                if spec["kind"] == "ctranspath":
                    stats = extract_ctranspath(ds, images_path, man, device)
                elif enc_name in ISOLATED_HF:
                    stats = extract_hf_isolated(ds, enc_name, spec, images_path, man, device)
                else:
                    stats = extract_inprocess(ds, enc_name, spec, images, man, device)
                msg = (f"ok  dim={stats['dim']}  eff_rank={stats['eff_rank']:.0f} "
                       f"({stats['eff_rank_frac']:.0%})  {(time.time() - t) / 60:.1f} min")
            except Exception as e:
                msg = f"SKIPPED: {type(e).__name__}: {str(e)[:300]}"
            print(f"  -> {enc_name}: {msg}", flush=True)
            log.append({"dataset": ds, "encoder": enc_name, "status": msg})
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass
    os.makedirs(CFG.LOG_DIR, exist_ok=True)
    with open(os.path.join(CFG.LOG_DIR, f"extract_{time.strftime('%Y%m%d_%H%M%S')}.json"), "w") as fh:
        json.dump(log, fh, indent=1)
    return log


def status_table(datasets, encoders):
    """Which (dataset, encoder) features exist and are complete."""
    import imgcache as IC
    rows = []
    for ds in datasets:
        jp = os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")
        man = json.load(open(jp)) if os.path.exists(jp) else None
        row = {"dataset": ds, "n": man["n"] if man else None}
        for e in encoders:
            row[e] = ("ok" if man and is_done(ds, e, man) else
                      ("partial" if os.path.isdir(_paths(ds, e)[3]) else "-"))
        rows.append(row)
    return rows


# ---------------------------------------------------------------------------
# consistency with the v2 features (pathology only)
# ---------------------------------------------------------------------------

V2_PREFERRED = ("trainval_features.npy", "all_features.npy", "features.npy")


def _find_v2(ds, enc):
    """Locate the v2 feature file covering the SAME rows as the v3 manifest.

    v3 fix: the previous version took the alphabetically first *features*.npy,
    which on BRACS was test_features.npy (the 570-image held-out institutions),
    so the comparison reported "row count differs" for UNI, Phikon-v2 and
    ResNet-50. Split-specific files (test_/train_/val_) are now never used."""
    if enc == "ctranspath":
        d = CFG.V2_CTRANSPATH_FIXED[ds]
    else:
        d = os.path.join(CFG.V2_CACHE[ds], CFG.V2_NAME[enc])
    if not os.path.isdir(d):
        return None, None
    present = set(os.listdir(d))
    fp = next((os.path.join(d, f) for f in V2_PREFERRED if f in present), None)
    if fp is None:
        cands = sorted(f for f in present if f.endswith(".npy") and "features" in f
                       and not f.startswith(("test_", "train_", "val_")))
        fp = os.path.join(d, cands[0]) if cands else None
    if fp is None:
        return None, None
    lp = os.path.join(d, os.path.basename(fp).replace("features", "labels"))
    return fp, (lp if os.path.exists(lp) else None)


def compare_with_v2(ds, enc, n_rows=3000):
    """Linear CKA between v3 and v2 features on the same rows.

    ~1.0 means the v3 pipeline reproduces v2's features for this encoder (the
    only differences are decode details). A low value means row order or
    preprocessing differs, which must be explained before using either set."""
    from cka import linear_cka
    if ds not in CFG.V2_CACHE or enc not in CFG.V2_NAME:
        return None
    fp, lp = _find_v2(ds, enc)
    _, npy, _, _ = _paths(ds, enc)
    if fp is None or not os.path.exists(npy):
        return {"dataset": ds, "encoder": enc, "status": "missing v2 or v3 features"}
    old = np.load(fp, mmap_mode="r")
    new = np.load(npy, mmap_mode="r")
    man = json.load(open(os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")))
    res = {"dataset": ds, "encoder": enc, "v2_file": os.path.relpath(fp, CFG.DRIVE),
           "n_v2": int(old.shape[0]), "n_v3": int(new.shape[0])}
    if old.shape[0] != new.shape[0]:
        res["status"] = "row count differs"
        return res
    if lp:
        y_old = np.load(lp)
        res["labels_identical"] = bool(np.array_equal(y_old, np.asarray(man["labels"])))
    idx = np.sort(np.random.default_rng(0).choice(new.shape[0], min(n_rows, new.shape[0]),
                                                  replace=False))
    res["cka_v2_v3"] = float(linear_cka(np.asarray(old[idx], np.float64),
                                        np.asarray(new[idx], np.float64)))
    res["status"] = ("reproduces v2" if res["cka_v2_v3"] > 0.98 else
                     "close to v2" if res["cka_v2_v3"] > 0.90 else
                     "DIFFERS FROM v2 - investigate before analysis")
    return res


# ---------------------------------------------------------------------------
# ResNet-50 provenance: which variant did v2's "resnet50_imagenet" use?
# Diagnostic only -- it does not change the registered v3 encoder.
# ---------------------------------------------------------------------------

RESNET_CANDIDATES = {
    "torchvision IMAGENET1K_V1 (v3)": ("timm", "resnet50.tv_in1k"),
    "torchvision IMAGENET1K_V2":      ("timm", "resnet50.tv2_in1k"),
    "timm a1_in1k":                   ("timm", "resnet50.a1_in1k"),
    "timm fb_swsl_ig1b_ft_in1k":      ("timm", "resnet50.fb_swsl_ig1b_ft_in1k"),
}


def resnet_provenance(ds="mhist", n_rows=1500, device="cuda"):
    """Compare v2's ResNet-50 features with several ImageNet ResNet-50 variants
    under two preprocessings, on the same rows.

    preprocessing "v3": the shared 224 cache (Resize 224 + CenterCrop 224)
    preprocessing "tv": torchvision's classic eval transform (Resize 256 +
                        CenterCrop 224) applied to the original images
    The row with CKA ~1.0 identifies what v2 used."""
    import torch
    from PIL import Image
    from cka import linear_cka
    import imgcache as IC
    fp, _ = _find_v2(ds, "resnet50_in1k")
    if fp is None:
        raise FileNotFoundError(f"no v2 ResNet-50 features for {ds}")
    old = np.load(fp, mmap_mode="r")
    images, man = IC.ensure_images(ds)
    idx = np.sort(np.random.default_rng(0).choice(man["n"], min(n_rows, man["n"]), replace=False))
    ref = np.asarray(old[idx], np.float64)

    def tv_batch(rows):
        out = []
        for i in rows:
            img = Image.open(man["reader"][1][i]).convert("RGB")
            w, h = img.size
            s = 256 / min(w, h)
            img = img.resize((max(256, round(w * s)), max(256, round(h * s))), Image.BILINEAR)
            W, H = img.size
            l, t = (W - 224) // 2, (H - 224) // 2
            out.append(np.asarray(img.crop((l, t, l + 224, t + 224)), np.uint8))
        return np.stack(out)

    can_tv = man["reader"][0] == "files"
    rows = []
    for label, (kind, mid) in RESNET_CANDIDATES.items():
        try:
            enc = E.build(label, dict(kind=kind, id=mid), device)
        except Exception as e:
            rows.append({"variant": label, "preprocessing": "-", "cka_with_v2": None,
                         "note": f"load failed: {type(e).__name__}"})
            continue
        for prep in (("v3", "tv") if can_tv else ("v3",)):
            feats = []
            for s in range(0, len(idx), 64):
                b = idx[s:s + 64]
                x = np.asarray(images[b]) if prep == "v3" else tv_batch(b)
                feats.append(enc(torch.from_numpy(np.ascontiguousarray(x))))
            Fn = np.concatenate(feats).astype(np.float64)
            rows.append({"variant": label, "preprocessing":
                         "Resize224+Crop224 (v3)" if prep == "v3" else "Resize256+Crop224",
                         "cka_with_v2": float(linear_cka(ref, Fn)), "note": ""})
        del enc
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return rows

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/ctranspath_worker.py
"""CTransPath feature extraction, run as a SEPARATE PROCESS under timm 0.5.4.

Why a separate process: CTransPath's released checkpoint matches the Swin
implementation of timm 0.5.4, which is the setup v2 verified (all 200 entries,
missing=0, unexpected=0, 16 ConvStem tensors). The other encoders need a modern
timm. Two timm versions cannot coexist in one Python process, so this script
runs with PYTHONPATH pointing at an isolated timm 0.5.4 install.

Hard guard: the script refuses to extract unless the checkpoint load reports
zero missing and zero unexpected keys AND the ConvStem tensors were loaded.
That is the exact check whose absence produced the v1 bug.

Usage (called by extract.extract_ctranspath):
    python ctranspath_worker.py --images X.npy --ckpt ctranspath.pth --shards DIR
"""

import argparse
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn


class ConvStem(nn.Module):
    """Convolutional patch embedding of CTransPath (Wang et al., 2022)."""

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768,
                 norm_layer=None, flatten=True):
        super().__init__()
        assert patch_size == 4 and embed_dim % 8 == 0
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.grid_size = (img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten
        stem, in_dim, out_dim = [], in_chans, embed_dim // 8
        for _ in range(2):
            stem += [nn.Conv2d(in_dim, out_dim, 3, 2, 1, bias=False),
                     nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True)]
            in_dim, out_dim = out_dim, out_dim * 2
        stem.append(nn.Conv2d(in_dim, embed_dim, 1))
        self.proj = nn.Sequential(*stem)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)
        return self.norm(x)


def build(ckpt):
    import timm
    if not timm.__version__.startswith("0.5."):
        raise RuntimeError(f"CTransPath worker must run under timm 0.5.x, got {timm.__version__}")
    m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=0)
    ed = getattr(m, "embed_dim", 96)
    if isinstance(ed, (list, tuple)):
        ed = ed[0]
    m.patch_embed = ConvStem(embed_dim=ed, norm_layer=nn.LayerNorm)
    # weights_only=False: torch>=2.6 defaults to True, which rejects this
    # (trusted, your own) checkpoint format.
    sd = torch.load(ckpt, map_location="cpu", weights_only=False)
    state = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    res = m.load_state_dict(state, strict=False)
    missing, unexpected = list(res.missing_keys), list(res.unexpected_keys)
    stem_keys = [k for k in state if k.startswith("patch_embed")]
    report = {"timm": timm.__version__, "torch": torch.__version__,
              "ckpt_entries": len(state), "missing": missing, "unexpected": unexpected,
              "patch_embed_tensors_in_ckpt": len(stem_keys)}
    print(f"  [ctranspath] timm {timm.__version__}  entries={len(state)}  "
          f"missing={len(missing)}  unexpected={len(unexpected)}  "
          f"ConvStem tensors={len(stem_keys)}", flush=True)
    if missing or unexpected or not stem_keys:
        raise SystemExit("INCOMPLETE CTransPath LOAD - refusing to extract "
                         f"(missing {missing[:5]}, unexpected {unexpected[:5]})")
    print("  [ctranspath] LOAD VERIFIED", flush=True)
    return m.eval(), report


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--shards", required=True)
    ap.add_argument("--shard_rows", type=int, default=2048)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--device", default="cuda")
    a = ap.parse_args()
    dev = a.device if (a.device != "cuda" or torch.cuda.is_available()) else "cpu"
    os.makedirs(a.shards, exist_ok=True)
    model, report = build(a.ckpt)
    model.to(dev)
    with open(os.path.join(a.shards, "load_report.json"), "w") as fh:
        json.dump(report, fh, indent=1)
    imgs = np.load(a.images, mmap_mode="r")
    n = imgs.shape[0]
    mean = torch.tensor((0.485, 0.456, 0.406), device=dev).view(1, 3, 1, 1)
    std = torch.tensor((0.229, 0.224, 0.225), device=dev).view(1, 3, 1, 1)
    ranges = [(s, min(s + a.shard_rows, n)) for s in range(0, n, a.shard_rows)]
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(a.shards, f"shard_{i:04d}.npy"))]
    print(f"  [ctranspath] {n} images, {len(todo)}/{len(ranges)} shards to go", flush=True)
    t0 = time.time()
    with torch.no_grad():                       # fp32, as in the v2 extraction
        for k, i in enumerate(todo):
            lo, hi = ranges[i]
            out = []
            for s in range(lo, hi, a.batch):
                x = torch.from_numpy(np.ascontiguousarray(imgs[s:min(s + a.batch, hi)]))
                x = x.to(dev).permute(0, 3, 1, 2).float().div_(255.0)
                f = model((x - mean) / std)
                if f.ndim == 3:
                    f = f.mean(1)
                out.append(f.float().cpu().numpy())
            np.save(os.path.join(a.shards, f"shard_{i:04d}.npy"),
                    np.concatenate(out).astype(np.float32))
            el = time.time() - t0
            print(f"      shard {i + 1}/{len(ranges)}  {hi}/{n}  "
                  f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/hf_worker.py
"""Feature extraction for a Hugging Face model whose remote code needs an OLDER
transformers than Colab ships (Hibou-B: its custom DINOv2 code imports helpers
removed in transformers 5).

Run as a separate process with PYTHONPATH pointing at an isolated install of
transformers 4.46 (see extract.ensure_old_transformers). torch, numpy and the
GPU come from the normal Colab environment. Output format is identical to the
in-process path: shard_XXXX.npy files plus a load report.

Usage (called by extract.extract_hf_isolated):
    python hf_worker.py --model histai/hibou-b --pool pooler --images X.npy --shards DIR
"""

import argparse
import json
import os
import time

import numpy as np
import torch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--pool", default="pooler", choices=["pooler", "cls"])
    ap.add_argument("--images", required=True)
    ap.add_argument("--shards", required=True)
    ap.add_argument("--shard_rows", type=int, default=2048)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--trust_remote_code", action="store_true")
    a = ap.parse_args()

    import transformers
    from transformers import AutoImageProcessor, AutoModel
    dev = a.device if (a.device != "cuda" or torch.cuda.is_available()) else "cpu"
    os.makedirs(a.shards, exist_ok=True)

    model = AutoModel.from_pretrained(a.model, trust_remote_code=a.trust_remote_code)
    model.eval().to(dev)
    try:
        proc = AutoImageProcessor.from_pretrained(a.model, trust_remote_code=a.trust_remote_code)
        mean, std = list(proc.image_mean), list(proc.image_std)
    except Exception:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    n_params = int(sum(p.numel() for p in model.parameters()))
    report = {"transformers": transformers.__version__, "torch": torch.__version__,
              "model": a.model, "class": type(model).__name__, "n_params": n_params,
              "mean": mean, "std": std, "pool": a.pool}
    print(f"  [{a.model}] transformers {transformers.__version__}  {type(model).__name__}  "
          f"{n_params / 1e6:.0f}M params  mean={mean} std={std}", flush=True)
    with open(os.path.join(a.shards, "load_report.json"), "w") as fh:
        json.dump(report, fh, indent=1)

    imgs = np.load(a.images, mmap_mode="r")
    n = imgs.shape[0]
    m_t = torch.tensor(mean, device=dev).view(1, 3, 1, 1)
    s_t = torch.tensor(std, device=dev).view(1, 3, 1, 1)
    ranges = [(s, min(s + a.shard_rows, n)) for s in range(0, n, a.shard_rows)]
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(a.shards, f"shard_{i:04d}.npy"))]
    print(f"  [{a.model}] {n} images, {len(todo)}/{len(ranges)} shards to go", flush=True)
    t0 = time.time()
    use_amp = dev.startswith("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        for k, i in enumerate(todo):
            lo, hi = ranges[i]
            out = []
            for s in range(lo, hi, a.batch):
                x = torch.from_numpy(np.ascontiguousarray(imgs[s:min(s + a.batch, hi)]))
                x = x.to(dev).permute(0, 3, 1, 2).float().div_(255.0)
                o = model(pixel_values=(x - m_t) / s_t)
                if a.pool == "pooler" and getattr(o, "pooler_output", None) is not None:
                    f = o.pooler_output
                else:
                    f = o.last_hidden_state[:, 0]
                out.append(f.float().cpu().numpy())
            np.save(os.path.join(a.shards, f"shard_{i:04d}.npy"),
                    np.concatenate(out).astype(np.float32))
            el = time.time() - t0
            print(f"      shard {i + 1}/{len(ranges)}  {hi}/{n}  "
                  f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/text_arm.py
"""Text arm: frozen sentence encoders on public classification sets.

Same downstream pipeline as the image arms (the analysis only ever sees an
(n, d) feature matrix and labels), so this adds independent pools in a
different modality at almost no engineering cost.
"""

from __future__ import annotations

import json
import os
import time

import numpy as np

import config as CFG
import imgcache as IC          # manifest_hash, stratified_cap
import extract as X            # _paths, is_done


def build_text_manifest(name, force=False):
    os.makedirs(CFG.MANIFEST_DIR, exist_ok=True)
    jp = os.path.join(CFG.MANIFEST_DIR, f"{name}.json")
    if os.path.exists(jp) and not force:
        with open(jp) as fh:
            return json.load(fh)
    from datasets import load_dataset
    hf_id, conf, split, tcol, lcol, cap = CFG.TEXT_DATASETS[name]
    ds = load_dataset(hf_id, conf, split=split) if conf else load_dataset(hf_id, split=split)
    texts = [str(t) for t in ds[tcol]]
    raw = list(ds[lcol])
    classes = sorted(set(raw))
    cmap = {c: i for i, c in enumerate(classes)}
    labels = np.array([cmap[v] for v in raw])
    keep_ok = np.array([len(t.strip()) > 0 for t in texts])
    idx_all = np.where(keep_ok)[0]
    sub = IC.stratified_cap(labels[idx_all], cap)
    keep = idx_all[sub]
    man = dict(name=name, n=int(len(keep)), n_before_cap=int(len(labels)), cap=cap,
               seed=CFG.SEED, classes=[str(c) for c in classes],
               uids=[f"{name}_{i}" for i in keep], labels=labels[keep].tolist(),
               groups=None, texts=[texts[i] for i in keep],
               class_counts=np.bincount(labels[keep], minlength=len(classes)).tolist(),
               source=f"{hf_id}:{conf or 'default'}:{split}", n_groups=None,
               built=time.strftime("%Y-%m-%d %H:%M:%S"))
    man["hash"] = IC.manifest_hash(man["uids"], man["labels"])
    with open(jp, "w") as fh:
        json.dump(man, fh)
    print(f"  [{name}] n={man['n']} classes={len(classes)}")
    return man


def run_text(datasets=None, encoders=None, device="cuda", batch=128):
    from sentence_transformers import SentenceTransformer
    import encoders as E
    datasets = datasets or list(CFG.TEXT_DATASETS)
    encoders = encoders or CFG.TEXT_ENCODERS
    mans = {}
    for ds in datasets:
        try:
            mans[ds] = build_text_manifest(ds)
        except Exception as e:
            print(f"  !! {ds}: manifest failed: {type(e).__name__}: {e}")
    log = []
    for enc_name, spec in encoders.items():          # load each model once
        todo = [ds for ds in mans if not X.is_done(ds, enc_name, mans[ds])]
        if not todo:
            print(f"{enc_name}: complete")
            continue
        t0 = time.time()
        try:
            model = SentenceTransformer(spec["id"], device=device)
        except Exception as e:
            print(f"  !! {enc_name}: load failed: {e}")
            continue
        for ds in todo:
            man = mans[ds]
            texts = [spec.get("prefix", "") + t for t in man["texts"]]
            F = model.encode(texts, batch_size=batch, show_progress_bar=False,
                             convert_to_numpy=True, normalize_embeddings=False)
            F = np.asarray(F, dtype=CFG.FEATURE_DTYPE)
            d, npy, meta, _ = X._paths(ds, enc_name)
            os.makedirs(d, exist_ok=True)
            np.save(npy, F)
            stats = E.feature_stats(F)
            with open(meta, "w") as fh:
                json.dump({"dataset": ds, "encoder": enc_name, "manifest_hash": man["hash"],
                           "shape": list(F.shape), "complete": True,
                           "encoder_info": {"name": enc_name, "kind": "st", "id": spec["id"],
                                            "prefix": spec.get("prefix", "")},
                           "stats": stats, "saved": time.strftime("%Y-%m-%d %H:%M:%S")}, fh, indent=1)
            print(f"  {enc_name:14s} {ds:14s} {F.shape}  eff_rank {stats['eff_rank']:.0f}")
            log.append((ds, enc_name, list(F.shape)))
        del model
        print(f"{enc_name}: done in {(time.time() - t0) / 60:.1f} min")
    return log

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("*** NO GPU. Runtime > Change runtime type > GPU (L4 or A100 recommended), then rerun. ***")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Choose what to run
Leave the lists as they are to run everything. To run part of it in one session, set `ONLY`
(datasets) and/or `SKIP_ENCODERS`. Completed (dataset, encoder) pairs are always skipped, so you can just re-run.

In [ ]:
import extract as X, imgcache as IC, encoders as E
for m in (X, IC, E):
    importlib.reload(m)
DATASETS = list(CFG.PATHOLOGY_DATASETS)
ENCODERS = dict(CFG.PATHOLOGY_ENCODERS)
ONLY = None             # e.g. ["bracs", "mhist"]
SKIP_ENCODERS = ()      # e.g. ("prov_gigapath",)
print("datasets:", DATASETS)
print("encoders:", list(ENCODERS))

## Status before running

In [ ]:
import pandas as pd
pd.set_option("display.width", 250)
pd.DataFrame(X.status_table(DATASETS, ENCODERS)).set_index("dataset")

## Extract
Per dataset: build the manifest, decode images once to 224 px, then run each encoder, writing a shard to Drive
every 2,048 images. If Colab disconnects, reconnect, run the first two cells, and run this cell again.
An encoder that fails (e.g. no licence) is reported and skipped; the others continue.

In [ ]:
log = X.run_domain(DATASETS, ENCODERS, device=DEVICE, only=ONLY, skip_encoders=SKIP_ENCODERS)
pd.DataFrame(log)

## Status after running

In [ ]:
pd.DataFrame(X.status_table(DATASETS, ENCODERS)).set_index("dataset")

## Feature sanity table
Effective rank, dead dimensions and a quick 3-fold probe accuracy (on ≤3,000 rows) for every completed pair.
An encoder whose features are degenerate (tiny effective rank, many dead dimensions, accuracy near the majority
baseline) stands out here — the v1 CTransPath bug would have shown up as exactly that.

In [ ]:
import json, numpy as np, folds as FO
rows = []
for ds in DATASETS:
    jp = os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")
    if not os.path.exists(jp):
        continue
    man = json.load(open(jp))
    y = np.asarray(man["labels"])
    idx = np.sort(np.random.default_rng(0).choice(len(y), min(3000, len(y)), replace=False))
    maj = np.bincount(y[idx]).max() / len(idx) * 100
    for e in ENCODERS:
        if not X.is_done(ds, e, man):
            continue
        meta = json.load(open(X._paths(ds, e)[2]))
        Fe = np.load(X._paths(ds, e)[1], mmap_mode="r")[idx]
        m = FO.make_folds(y[idx], k=3, seed=0)
        acc = FO.accuracy(FO.per_fold_logits(np.asarray(Fe), y[idx], m, backend="torch"), y[idx]) * 100
        s = meta["stats"]
        rows.append(dict(dataset=ds, encoder=e, dim=s["dim"], eff_rank=round(s["eff_rank"]),
                         eff_rank_frac=round(s["eff_rank_frac"], 3), dead=round(s["dead_dims_frac"], 3),
                         quick_acc=round(acc, 2), majority=round(maj, 2)))
sanity = pd.DataFrame(rows)
sanity.to_csv(os.path.join(CFG.RESULTS_DIR, f"feature_sanity_{DATASETS[0]}.csv"), index=False)
sanity

Compact view (nothing truncated): quick probe accuracy, encoder × dataset, with the majority baseline as the last row.

In [ ]:
piv = sanity.pivot(index="encoder", columns="dataset", values="quick_acc").round(1)
piv.loc["(majority)"] = sanity.groupby("dataset")["majority"].first().round(1)
piv

## Consistency with your v2 features
For BRACS, SICAPv2 and MHIST, compares the new UNI, Phikon-v2, CTransPath and ResNet-50 features with the cached
v2 ones (same rows) by linear CKA. ~1.0 means v3 reproduces v2. Lower values mean the row order or preprocessing
differs (v3 decodes once with Resize+CenterCrop 224 for every encoder; if v2 used a different transform for an
encoder, expect 0.9–0.98). Below 0.9 must be explained before analysis. For ResNet-50, v3 uses torchvision's
ImageNet weights; if v2 used different weights this will show here.

In [ ]:
cmp = [X.compare_with_v2(ds, e) for ds in ("bracs", "sicapv2", "mhist")
       for e in ("uni", "phikon_v2", "ctranspath", "resnet50_in1k")]
cmp = pd.DataFrame([c for c in cmp if c])
cmp.to_csv(os.path.join(CFG.RESULTS_DIR, "v2_consistency.csv"), index=False)
cmp

## Which ResNet-50 did v2 use? (diagnostic only)
v3's pathology ResNet-50 is torchvision `IMAGENET1K_V1`; on SICAPv2 and MHIST it matched v2's `resnet50_imagenet`
features at CKA 0.65 / 0.74 while every other encoder matched at 0.9999+. This cell compares v2's features with four
ImageNet ResNet-50 variants under two preprocessings on 1,500 MHIST images. The row at CKA ≈ 1.0 identifies what v2
used. **This does not change the analysis**: the registered v3 encoder stays `IMAGENET1K_V1`; the result is for
the paper's methods section (one sentence stating which ResNet-50 v2 used and that v3 standardises on V1).
About 3 minutes on a GPU.

In [ ]:
prov = pd.DataFrame(X.resnet_provenance("mhist", n_rows=1500, device=DEVICE))
prov.to_csv(os.path.join(CFG.RESULTS_DIR, "resnet50_provenance_mhist.csv"), index=False)
prov.sort_values("cka_with_v2", ascending=False)

## Free local disk (optional)
Deletes the local image caches and downloads. The Drive-persisted caches (BRACS, SICAPv2, MHIST) and all
features stay on Drive.

In [ ]:
import shutil
for ds in DATASETS:
    IC.drop_local_images(ds)
shutil.rmtree(CFG.LOCAL_DL, ignore_errors=True)
print("local caches removed")